## Concept primer: answering business questions with pandas

| Question shape | Code |
|---|---|
| "what % defaulted?" | `df["defaulted"].mean()`: the **mean of a 0/1 column is the rate** |
| "rate per group" | `df.groupby("region")["defaulted"].mean()` |
| several stats per group | `df.groupby("region")["defaulted"].agg(["mean", "count"])` |
| highest group | `.sort_values(ascending=False)` then `.index[0]` (name) / `.iloc[0]` (value); or `.idxmax()` / `.max()` |
| share of each category | `df["region"].value_counts(normalize=True)` |
| per month | `df.set_index("date_col")["x"].resample("ME").count()` or `df["date_col"].dt.to_period("M")` then groupby |
| new column | `df["ratio"] = df["a"] / df["b"]` (vectorised, no loop) |
| split into equal-sized bands | `pd.qcut(series, 3, labels=["low", "medium", "high"])` (equal **counts**; `pd.cut` = equal **widths**) |

**Always report the group size (`count`) next to a rate.** 3 defaults out of 5 is 60% but means almost nothing.

## Problem: loan default analysis

`data/loans_clean.csv` is the cleaned output of drill 05 (same steps, already done for you).
Answer each question in code, store the answer in the named variable, and run `check(...)`.
The check cell tells you CORRECT/WRONG without revealing the answer.

**Target: 25 minutes.**

In [1]:
import pandas as pd
df = pd.read_csv("../data/loans_clean.csv", parse_dates=["application_date"])
df.head()


,application_id,application_date,region,employment,income,loan_amount,defaulted
0,A10374,2025-12-26,Leeds,self-employed,24600.0,15800.0,1
1,A10267,2025-07-24,Leeds,unemployed,34600.0,16800.0,0
2,A10372,2025-05-31,London,employed,56400.0,16400.0,1
3,A10161,2025-02-15,Glasgow,employed,38700.0,14800.0,1
4,A10327,2025-01-23,London,unemployed,69100.0,2100.0,1


In [2]:
import hashlib
_ANS = {"q1": "cd65e6f2e1", "q2": "317532540a", "q2r": "cf3567ceb9", "q3": "d52dfb3e2c",
        "q4": "a5de54214e", "q4n": "642e92efb7", "q5": "2c4e0075d5"}
def check(key, value):
    """Compares your answer to the hidden one without showing it. Floats are rounded to 3 dp."""
    if isinstance(value, float): value = round(value, 3)
    if hasattr(value, "item"): value = value.item()          # numpy/pandas scalar -> python
    if isinstance(value, float): value = round(value, 3)
    ok = hashlib.md5(str(value).encode()).hexdigest()[:10] == _ANS[key]
    print(f"{key}: {'CORRECT' if ok else 'WRONG'}  (you gave {value!r})")


**Q1.** What is the overall default rate? (a proportion between 0 and 1)

In [3]:
q1 = round(df["defaulted"].mean(), 3)
q1

np.float64(0.264)

In [4]:
check("q1", q1)


q1: CORRECT  (you gave 0.264)


**Q2.** Which region has the highest default rate, and what is that rate?

In [5]:
rates = df.groupby("region")["defaulted"].mean().sort_values(ascending=False)
top_region = rates.index[0]
top_region_rate = round(rates.iloc[0], 3)
rates.round(3)

region
Manchester    0.333
London        0.241
Leeds         0.240
Glasgow       0.231
Name: defaulted, dtype: float64

In [6]:
check("q2", top_region)
check("q2r", top_region_rate)


q2: CORRECT  (you gave 'Manchester')
q2r: CORRECT  (you gave 0.333)


**Q3.** Which employment status has the highest default rate? Show the **count** per group too. Is any group too small to trust?

In [7]:
rates = df.groupby("employment")["defaulted"].mean().sort_values(ascending=False)
counts = df["employment"].value_counts()
top_employment = rates.index[0]
pd.DataFrame({"default_rate": rates.round(3), "n": counts})

,default_rate,n
employment,,
employed,0.254,311
self-employed,0.239,88
unemployed,0.347,49
unknown,0.308,26


In [8]:
check("q3", top_employment)


q3: CORRECT  (you gave 'unemployed')


**Q4.** Which month had the most applications, and how many? Give the month as a string like `"2025-03"`.
(Rows with a missing date can't be placed in a month: leave them out.)

In [9]:
value_counts = df["application_date"].dropna().dt.strftime("%Y-%m").value_counts()
busiest_month = value_counts.index[0]
busiest_month_n = int(value_counts.iloc[0])
value_counts.head()

application_date
2025-12    48
2025-05    46
2025-08    44
2025-01    42
2025-04    42
Name: count, dtype: int64

In [10]:
check("q4", busiest_month)
check("q4n", busiest_month_n)


q4: CORRECT  (you gave '2025-12')
q4n: CORRECT  (you gave 48)


**Q5.** Create `loan_to_income = loan_amount / income`. Split it into 3 equal-sized bands (`low`, `medium`, `high`) with `pd.qcut`.
What is the default rate in the **high** band? Look at all three bands: what's the pattern?

In [11]:
df["loan_to_income"] = df["loan_amount"] / df["income"]
df["lti_band"] = pd.qcut(df["loan_to_income"], 3, labels=["low", "medium", "high"])
band_rates = df.groupby("lti_band", observed=True)["defaulted"].mean()
high_band_rate = round(band_rates["high"], 3)
band_rates.round(3)    # low 0.152, medium 0.247, high 0.392: default rises with loan-to-income

lti_band
low       0.152
medium    0.247
high      0.392
Name: defaulted, dtype: float64

In [12]:
check("q5", high_band_rate)


q5: CORRECT  (you gave 0.392)


## Explain it (graded)

1. **Recommendation:** in 2 lines, what would you tell the credit-risk team based on Q2-Q5?
2. **Caveats:** name 2 reasons these results might mislead (think: group sizes, how missing values were filled, correlation vs causation).

✅ **Model answer**

1. **Recommendation:** loan-to-income is the strongest signal: the high band defaults at 39% against 15% in the low band, so tighten limits or pricing for high loan-to-income applications. Manchester (33% vs about 24% elsewhere) and unemployed applicants (35%) also deserve a closer look.
2. **Caveats:** (a) **group sizes**: unemployed is only 49 applicants and unknown 26, so their rates are noisy (wide confidence intervals); (b) missing incomes were **filled with the median**, which shrinks real differences in loan-to-income; (c) these are **correlations**, not causes: Manchester's rate may reflect a different mix of employment or loan sizes, so check it within bands before acting on region.